# Minimal MATLAB-branch probe for one pipeline

This notebook builds **one** MATLAB-backed pipeline input bundle from your current `fnirs_benchmark_v7.py`, then runs the MATLAB helper on that bundle **outside** the full batch script.

It is intentionally narrow:

- one subject
- one file condition
- one MATLAB pipeline

It uses the same Python functions your script uses for:
- config construction
- preprocessing
- MATLAB bundle creation

Then it runs the MATLAB helper in a **separate MATLAB process** from the notebook, which is safer than invoking the Engine directly from the notebook kernel when MATLAB is unstable.

Once this works, you can decide whether you still need to debug the Engine wrapper separately.


In [1]:
from pathlib import Path
import sys
import os
import json
import time
import traceback
import importlib.util
import subprocess

import numpy as np
import pandas as pd
import mne

# ---- EDIT THESE IF NEEDED ----
SCRIPT_PATH = Path("/home/asunkari/fnirs-representation-learning/v2/fnirs_benchmark_v7.py")
ROOT = Path("/home/asunkari/fnirs-representation-learning")
TRUTH_TEMPLATE_DIR = ROOT / "synthetic_hrf_generation"
ANALYZIR_PATH = Path("/home/asunkari/nirs-toolbox")
SUBJECT = "Subj94"
FILE_LABEL = "hrf_20"
PIPELINE_LABEL = "LocalSS_Glover_ARIRLS"
MATLAB_CMD = "/home/asunkari/matlab/bin/matlab.sh"

for path_obj in [SCRIPT_PATH, ROOT, TRUTH_TEMPLATE_DIR, ANALYZIR_PATH]:
    assert path_obj.exists(), path_obj

print("Using script:", SCRIPT_PATH)
print("Using root:", ROOT)
print("Using truth templates:", TRUTH_TEMPLATE_DIR)
print("Using AnalyzIR path:", ANALYZIR_PATH)
print("Using subject/file/pipeline:", SUBJECT, FILE_LABEL, PIPELINE_LABEL)
print("Using MATLAB command:", MATLAB_CMD)


Using script: /home/asunkari/fnirs-representation-learning/v2/fnirs_benchmark_v7.py
Using root: /home/asunkari/fnirs-representation-learning
Using truth templates: /home/asunkari/fnirs-representation-learning/synthetic_hrf_generation
Using AnalyzIR path: /home/asunkari/nirs-toolbox
Using subject/file/pipeline: Subj94 hrf_20 LocalSS_Glover_ARIRLS
Using MATLAB command: /home/asunkari/matlab/bin/matlab.sh


In [2]:
spec = importlib.util.spec_from_file_location("bench", str(SCRIPT_PATH))
bench = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = bench
spec.loader.exec_module(bench)

print("Loaded module:", bench.__name__)
print("MATLAB Engine available:", bench.optional_import_matlab_engine()[1] is not None)


Loaded module: bench
MATLAB Engine available: True


In [3]:
config = bench.BenchmarkConfig(
    root=str(ROOT),
    truth_template_dir=str(TRUTH_TEMPLATE_DIR),
    analyzir_path=str(ANALYZIR_PATH),
    use_matlab=True,
    use_matlab_engine=False,      # safer first probe: MATLAB process outside notebook kernel
    prefer_matlab_engine=False,   # keep this false for the notebook probe
    matlab_cmd=MATLAB_CMD,
    matlab_startup_options="",    # do not pass -nojvm here
    empirical_null_shift_count=0, # active-condition probe only; no null shifts
    n_workers=1,
    overwrite=True,
)
config.file_specs = bench.default_file_specs()
config.pipeline_specs = bench.default_pipeline_specs()

file_spec = next(fs for fs in config.file_specs if fs.label == FILE_LABEL)
pipeline = next(ps for ps in config.pipeline_specs if ps.label == PIPELINE_LABEL)

print("FileSpec:", file_spec)
print("PipelineSpec:", pipeline)

pd.DataFrame([bench.asdict(p) for p in config.pipeline_specs]).query("label == @PIPELINE_LABEL")


FileSpec: FileSpec(label='hrf_20', filename='resting_hrf_20.snirf', amplitude_value=20, is_null=False, annotation_source_filename=None)
PipelineSpec: PipelineSpec(label='LocalSS_Glover_ARIRLS', backend='matlab_arirls', nuisance_method='local_nearest', hrf_model='glover', solver='arirls', pruning_style='strict_combined', motion_method='tddr', filter_mode='bandpass', use_block_average=False, include_in_empirical_null=True, include_in_primary_variability=True, secondary_pipeline=False, comparison_group='core', description='StrictQC -> TDDR -> band-pass -> LocalSS -> Glover -> AR-IRLS')


,label,backend,nuisance_method,hrf_model,solver,pruning_style,motion_method,filter_mode,use_block_average,include_in_empirical_null,include_in_primary_variability,secondary_pipeline,comparison_group,description
8,LocalSS_Glover_ARIRLS,matlab_arirls,local_nearest,glover,arirls,strict_combined,tddr,bandpass,False,True,True,False,core,StrictQC -> TDDR -> band-pass -> LocalSS -> Gl...


## Recreate the early job state in the notebook

This mirrors the same data-loading / truth-label / QC steps the batch script uses. fileciteturn70file0

In [4]:
def prepare_job_state(file_label: str):
    file_spec = next(fs for fs in config.file_specs if fs.label == file_label)
    dataset_dir = config.dataset_path()
    subject_dir = dataset_dir / SUBJECT
    snirf_file_path = subject_dir / file_spec.filename
    annotation_source_path = subject_dir / file_spec.annotation_source_filename if file_spec.annotation_source_filename else None
    reference_path = subject_dir / "resting_hrf_20.snirf"

    if not snirf_file_path.exists():
        raise FileNotFoundError(snirf_file_path)
    if not reference_path.exists():
        raise FileNotFoundError(reference_path)

    raw_cw = mne.io.read_raw_snirf(snirf_file_path, preload=True, verbose=False)
    if file_spec.is_null:
        raw_cw = bench.copy_valid_annotations(raw_cw, annotation_source_path)
    raw_cw = bench.sanitize_annotations_to_single_task(raw_cw)

    reference_raw = mne.io.read_raw_snirf(reference_path, preload=True, verbose=False)
    data_type_labels = bench.read_measurement_data_type_labels(reference_path)
    if data_type_labels is None:
        raise RuntimeError("Could not read truth labels from reference file.")

    picks_cw = bench.get_cw_channel_indices(reference_raw)
    cw_names = np.asarray(reference_raw.ch_names)[picks_cw]
    if len(data_type_labels) == len(cw_names):
        aligned_names = cw_names
    elif len(data_type_labels) == len(reference_raw.ch_names):
        aligned_names = np.asarray(reference_raw.ch_names)
    else:
        raise RuntimeError("Truth-label alignment failed.")

    target_pair_names = sorted(set(name.split(" ")[0] for name in aligned_names[data_type_labels == 1].tolist()))
    target_pair_set = set(target_pair_names)

    cw_channel_table = bench.build_cw_channel_table(reference_raw, SUBJECT, file_spec.label, config)
    long_pair_names = sorted(cw_channel_table.loc[cw_channel_table["group"] == "LS", "pair_name"].astype(str).unique())
    non_target_pair_names = [pair for pair in long_pair_names if pair not in target_pair_set]

    channel_quality, pair_quality = bench.build_quality_tables(raw_cw, SUBJECT, file_spec.label, config)
    truth_templates = bench.load_truth_templates(config)

    return {
        "file_spec": file_spec,
        "subject_dir": subject_dir,
        "snirf_file_path": snirf_file_path,
        "raw_cw": raw_cw,
        "reference_raw": reference_raw,
        "target_pair_names": target_pair_names,
        "target_pair_set": target_pair_set,
        "non_target_pair_names": non_target_pair_names,
        "cw_channel_table": cw_channel_table,
        "channel_quality": channel_quality,
        "pair_quality": pair_quality,
        "truth_templates": truth_templates,
    }

state = prepare_job_state(FILE_LABEL)
print("Annotations:", len(state["raw_cw"].annotations))
print("True target pairs:", state["target_pair_names"])
print("n true targets:", len(state["target_pair_names"]))
print("n true non-target pairs:", len(state["non_target_pair_names"]))
state["pair_quality"].head()


/tmp/ipykernel_614616/4243253308.py:14: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_cw = mne.io.read_raw_snirf(snirf_file_path, preload=True, verbose=False)
/tmp/ipykernel_614616/4243253308.py:19: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage()

Annotations: 34
True target pairs: ['S11_D25', 'S3_D2', 'S9_D17', 'S9_D23']
n true targets: 4
n true non-target pairs: 44


,subject,file_label,pair_name,sci_min,sci_mean,snr_min,snr_mean,negative_fraction_max,distance_m,group,midpoint_x,midpoint_y,midpoint_z,hemisphere
0,Subj94,hrf_20,S10_D17,0.371336,0.371336,3.315046,8.359204,0.0,0.030017,LS,-0.1425,-0.013,0.0,left
1,Subj94,hrf_20,S10_D18,0.299605,0.299605,3.619085,10.515340,0.0,0.030017,LS,-0.1575,-0.013,0.0,left
2,Subj94,hrf_20,S10_D21,0.193980,0.193980,2.658280,11.073409,0.0,0.008000,SS,-0.1500,0.004,0.0,left
3,Subj94,hrf_20,S10_D23,0.400249,0.400249,3.274104,17.934226,0.0,0.030017,LS,-0.1425,0.013,0.0,left
4,Subj94,hrf_20,S10_D24,0.231181,0.231181,2.781529,10.137340,0.0,0.030017,LS,-0.1575,0.013,0.0,left


## Build the single-pipeline MATLAB bundle using the script's own functions

This uses:
- `preprocess_raw_to_hb(...)`
- `execute_python_pipeline(...)`
- `write_matlab_bundle(...)`

exactly as the batch script does for MATLAB-backed pipelines. fileciteturn70file0

In [5]:
raw_hb, bad_pairs = bench.preprocess_raw_to_hb(
    state["raw_cw"],
    state["pair_quality"],
    pipeline,
    config,
)

result = bench.execute_python_pipeline(
    subject=SUBJECT,
    file_spec=state["file_spec"],
    pipeline=pipeline,
    raw_cw=state["raw_cw"],
    raw_hb=raw_hb,
    target_pair_names=state["target_pair_set"],
    truth_templates=state["truth_templates"],
    snirf_file_path=state["snirf_file_path"],
    config=config,
)

print("bad pairs:", len(bad_pairs))
print("matlab_input_specs_list:", len(result["matlab_input_specs_list"]))
print("matlab_shift_specs_list:", len(result["matlab_shift_specs_list"]))
print("nuisance_detail rows:", len(result["nuisance_detail"]))
if isinstance(result["nuisance_detail"], pd.DataFrame) and len(result["nuisance_detail"]) > 0:
    display(result["nuisance_detail"].head())


/home/asunkari/fnirs-representation-learning/v2/fnirs_benchmark_v7.py:1022: RuntimeWarning: Negative intensities encountered. Setting to abs(x)
  raw_od = optical_density(raw_cw.copy())


bad pairs: 49
matlab_input_specs_list: 1
matlab_shift_specs_list: 0
nuisance_detail rows: 12


,subject,file_label,pipeline_label,channel_name,chromophore,nuisance_method_used,nuisance_regressor_label,nearest_short_distance_m,n_short_channels_used
0,Subj94,hrf_20,LocalSS_Glover_ARIRLS,S13_D26 hbo,hbo,pooled_fallback_average,pooled_fallback_average,0.271184,1
1,Subj94,hrf_20,LocalSS_Glover_ARIRLS,S13_D26 hbr,hbr,pooled_fallback_average,pooled_fallback_average,0.271184,1
2,Subj94,hrf_20,LocalSS_Glover_ARIRLS,S14_D31 hbo,hbo,pooled_fallback_average,pooled_fallback_average,0.309631,1
3,Subj94,hrf_20,LocalSS_Glover_ARIRLS,S14_D31 hbr,hbr,pooled_fallback_average,pooled_fallback_average,0.309631,1
4,Subj94,hrf_20,LocalSS_Glover_ARIRLS,S5_D4 hbo,hbo,pooled_fallback_average,pooled_fallback_average,0.024418,1


In [6]:
job_dir = config.output_path() / "notebook_single_matlab_probe" / SUBJECT / FILE_LABEL / PIPELINE_LABEL
job_dir.mkdir(parents=True, exist_ok=True)

bundle_path = bench.write_matlab_bundle(
    result["matlab_input_specs_list"],
    result["matlab_shift_specs_list"],
    job_dir,
)

print("Bundle path:", bundle_path)
bundle = json.loads(bundle_path.read_text())
print(json.dumps(bundle, indent=2)[:1200])


Bundle path: /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_single_matlab_probe/Subj94/hrf_20/LocalSS_Glover_ARIRLS/matlab_inputs/matlab_bundle.json
{
  "input_mat_files": [
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_single_matlab_probe/Subj94/hrf_20/LocalSS_Glover_ARIRLS/matlab_inputs/0001__LocalSS_Glover_ARIRLS__observed__input.mat"
  ],
  "output_csv_files": [
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_single_matlab_probe/Subj94/hrf_20/LocalSS_Glover_ARIRLS/matlab_inputs/0001__LocalSS_Glover_ARIRLS__observed__output.csv"
  ]
}


## Run the MATLAB helper in a separate MATLAB process

This is the safest notebook-level probe because:
- it does **not** run the whole batch script
- it exercises the **MATLAB branch**
- if MATLAB fails, it should not take down the notebook kernel the way an in-kernel Engine call can

It uses the same helper `.m` file that your script uses. fileciteturn70file0

In [16]:
helper_m = SCRIPT_PATH.with_name("analyzir_arirls_batch.m")
assert helper_m.exists(), helper_m
print(helper_m)

/home/asunkari/fnirs-representation-learning/v2/analyzir_arirls_batch.m


In [17]:
_, matlab_engine = bench.optional_import_matlab_engine()
assert matlab_engine is not None, "MATLAB Engine is not available in this Python env"

eng = matlab_engine.start_matlab(config.matlab_startup_options)
helper_dir = helper_m.parent.as_posix()

eng.addpath(helper_dir, nargout=0)
eng.setenv("FNIRS_BUNDLE_JSON", str(bundle_path), nargout=0)
eng.setenv("FNIRS_ANALYZIR_PATH", str(config.analyzir_path), nargout=0)

print("helper:", eng.eval("which('analyzir_arirls_batch')", nargout=1))
print("GLM:", eng.eval("which('nirs.modules.GLM')", nargout=1))
print("Resample:", eng.eval("which('nirs.modules.Resample')", nargout=1))
print("AR_IRLS:", eng.eval("which('nirs.modules.AR_IRLS')", nargout=1))
print("Dictionary:", eng.eval("which('Dictionary')", nargout=1))

helper: /home/asunkari/fnirs-representation-learning/v2/analyzir_arirls_batch.m
GLM: /home/asunkari/nirs-toolbox/+nirs/+modules/GLM.m
Resample: /home/asunkari/nirs-toolbox/+nirs/+modules/Resample.m
AR_IRLS: /home/asunkari/nirs-toolbox/+nirs/+modules/AR_IRLS.m
Dictionary: /home/asunkari/nirs-toolbox/external/Dictionary/Dictionary.m


In [19]:
_, matlab_engine = bench.optional_import_matlab_engine()
assert matlab_engine is not None, "MATLAB Engine is not available in this Python env"

eng = matlab_engine.start_matlab(config.matlab_startup_options)
helper_dir = helper_m.parent.as_posix()

eng.addpath(helper_dir, nargout=0)
eng.setenv("FNIRS_BUNDLE_JSON", str(bundle_path), nargout=0)
eng.setenv("FNIRS_ANALYZIR_PATH", str(config.analyzir_path), nargout=0)

print("helper:", eng.eval("which('analyzir_arirls_batch')", nargout=1))
print("GLM:", eng.eval("which('nirs.modules.GLM')", nargout=1))
print("Resample:", eng.eval("which('nirs.modules.Resample')", nargout=1))
print("AR_IRLS:", eng.eval("which('nirs.modules.AR_IRLS')", nargout=1))
print("Dictionary:", eng.eval("which('Dictionary')", nargout=1))

helper: /home/asunkari/fnirs-representation-learning/v2/analyzir_arirls_batch.m
GLM: /home/asunkari/nirs-toolbox/+nirs/+modules/GLM.m
Resample: /home/asunkari/nirs-toolbox/+nirs/+modules/Resample.m
AR_IRLS: /home/asunkari/nirs-toolbox/+nirs/+modules/AR_IRLS.m
Dictionary: /home/asunkari/nirs-toolbox/external/Dictionary/Dictionary.m


In [20]:
out = eng.evalc(r"""
cfg = jsondecode(fileread(getenv('FNIRS_BUNDLE_JSON')));
disp(cfg.input_mat_files{1});
S = load(string(cfg.input_mat_files{1}));

disp('--- fieldnames(S) ---');
disp(fieldnames(S));

disp('--- size(S.Y) ---');
disp(size(S.Y));

if isfield(S,'nuisance_values')
    disp('--- size(S.nuisance_values) ---');
    disp(size(S.nuisance_values));
end

disp('--- first channel ---');
disp(S.channel_names(1));
disp(S.chromophores(1));
disp(S.pipeline_label);
""", nargout=1)

print(out)

/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/notebook_single_matlab_probe/Subj94/hrf_20/LocalSS_Glover_ARIRLS/matlab_inputs/0001__LocalSS_Glover_ARIRLS__observed__input.mat
--- fieldnames(S) ---
    {'times_s'           }
    {'stim_onsets_s'     }
    {'stim_durations_s'  }
    {'stim_amplitudes'   }
    {'Y'                 }
    {'X'                 }
    {'n_reg'             }
    {'task_reg_index'    }
    {'nuisance_values'   }
    {'nuisance_n_reg'    }
    {'nuisance_names'    }
    {'matlab_resample_fs'}
    {'channel_names'     }
    {'pair_names'        }
    {'chromophores'      }
    {'target_status'     }
    {'subject'           }
    {'file_label'        }
    {'pipeline_label'    }
    {'backend'           }
    {'hrf_model'         }
    {'solver'            }
    {'amplitude_value'   }
    {'shift_index'       }
    {'shift_s'           }

--- size(S.Y) ---
       34650          12

--- size(S.nuisance_values) ---
       34650           1        

In [21]:
out = eng.evalc(r"""
try
    cfg = jsondecode(fileread(getenv('FNIRS_BUNDLE_JSON')));
    S = load(string(cfg.input_mat_files{1}));

    ch = 1;
    times_s = double(S.times_s(:));
    stim_onsets_s = double(S.stim_onsets_s(:));
    stim_durations_s = double(S.stim_durations_s(:));
    stim_amplitudes = double(S.stim_amplitudes(:));
    yi = double(S.Y(:, ch));
    chrom = char(string(S.chromophores{ch}));

    nuisance_values = [];
    nuisance_n_reg = [];
    nuisance_names = {};
    if isfield(S, 'nuisance_values')
        nuisance_values = double(S.nuisance_values);
    end
    if isfield(S, 'nuisance_n_reg')
        nuisance_n_reg = double(S.nuisance_n_reg(:));
    end
    if isfield(S, 'nuisance_names')
        nuisance_names = S.nuisance_names;
    end

    probe = nirs.core.Probe(zeros(1,3), zeros(1,3), ...
        table(1, 1, {char(chrom)}, 'VariableNames', {'source','detector','type'}));
    data_obj = nirs.core.Data(yi(:), times_s(:), probe);

    task = nirs.design.StimulusEvents();
    task.name = 'task';
    task.onset = stim_onsets_s(:);
    task.dur = stim_durations_s(:);
    task.amp = stim_amplitudes(:);
    data_obj.stimulus('task') = task;

    if ~isempty(nuisance_values) && ~isempty(nuisance_n_reg)
        n_nuis = nuisance_n_reg(ch);
        for k = 1:n_nuis
            reg_vec = double(squeeze(nuisance_values(:, k, ch)));
            if isempty(reg_vec)
                continue;
            end
            reg_name = sprintf('nuis_%02d', k);
            stim_vec = nirs.design.StimulusVector();
            stim_vec.name = reg_name;
            stim_vec.time = times_s(:);
            stim_vec.vector = reg_vec(:);
            try
                stim_vec.regressor_no_interest = true;
            catch
            end
            data_obj.stimulus(reg_name) = stim_vec;
        end
    end

    disp('--- built data_obj ---');
    disp(class(data_obj));
    disp(size(data_obj.data));

    resample_fs = 4.0;
    if isfield(S, 'matlab_resample_fs')
        resample_fs = double(S.matlab_resample_fs(1));
    end

    resample_job = nirs.modules.Resample();
    resample_job.Fs = resample_fs;
    data_rs = resample_job.run(data_obj);

    disp('--- after resample ---');
    disp(class(data_rs));
    disp(size(data_rs.data));

catch ME
    disp(getReport(ME, 'extended'));
end
""", nargout=1)

print(out)

--- built data_obj ---
nirs.core.Data
       34650           1

--- after resample ---
nirs.core.Data
        2772           1




In [22]:
out = eng.evalc(r"""
try
    cfg = jsondecode(fileread(getenv('FNIRS_BUNDLE_JSON')));
    S = load(string(cfg.input_mat_files{1}));

    ch = 1;
    times_s = double(S.times_s(:));
    stim_onsets_s = double(S.stim_onsets_s(:));
    stim_durations_s = double(S.stim_durations_s(:));
    stim_amplitudes = double(S.stim_amplitudes(:));
    yi = double(S.Y(:, ch));
    chrom = char(string(S.chromophores{ch}));

    nuisance_values = [];
    nuisance_n_reg = [];
    nuisance_names = {};
    if isfield(S, 'nuisance_values')
        nuisance_values = double(S.nuisance_values);
    end
    if isfield(S, 'nuisance_n_reg')
        nuisance_n_reg = double(S.nuisance_n_reg(:));
    end
    if isfield(S, 'nuisance_names')
        nuisance_names = S.nuisance_names;
    end

    probe = nirs.core.Probe(zeros(1,3), zeros(1,3), ...
        table(1, 1, {char(chrom)}, 'VariableNames', {'source','detector','type'}));
    data_obj = nirs.core.Data(yi(:), times_s(:), probe);

    task = nirs.design.StimulusEvents();
    task.name = 'task';
    task.onset = stim_onsets_s(:);
    task.dur = stim_durations_s(:);
    task.amp = stim_amplitudes(:);
    data_obj.stimulus('task') = task;

    if ~isempty(nuisance_values) && ~isempty(nuisance_n_reg)
        n_nuis = nuisance_n_reg(ch);
        for k = 1:n_nuis
            reg_vec = double(squeeze(nuisance_values(:, k, ch)));
            if isempty(reg_vec)
                continue;
            end
            reg_name = sprintf('nuis_%02d', k);
            stim_vec = nirs.design.StimulusVector();
            stim_vec.name = reg_name;
            stim_vec.time = times_s(:);
            stim_vec.vector = reg_vec(:);
            try
                stim_vec.regressor_no_interest = true;
            catch
            end
            data_obj.stimulus(reg_name) = stim_vec;
        end
    end

    resample_fs = 4.0;
    if isfield(S, 'matlab_resample_fs')
        resample_fs = double(S.matlab_resample_fs(1));
    end

    resample_job = nirs.modules.Resample();
    resample_job.Fs = resample_fs;
    data_rs = resample_job.run(data_obj);

    glm_job = nirs.modules.GLM();
    glm_job.type = 'AR-IRLS';

    basis_dict = Dictionary();
    hrf_model = lower(char(string(S.hrf_model)));
    switch hrf_model
        case {'glover','canonical'}
            basis_dict('default') = nirs.design.basis.Canonical();
        case 'gamma'
            basis_dict('default') = nirs.design.basis.Gamma();
        otherwise
            error('Unsupported MATLAB HRF model for module workflow: %s', hrf_model);
    end
    glm_job.basis = basis_dict;

    disp('--- about to run glm_job.run(data_rs) ---');
    Stats = glm_job.run(data_rs);

    disp('--- Stats class ---');
    disp(class(Stats));
    disp('--- Stats.beta ---');
    disp(Stats.beta);

catch ME
    disp(getReport(ME, 'extended'));
end
""", nargout=1)

print(out)

--- about to run glm_job.run(data_rs) ---
High collinearity: cond(X) = 9291.4322.
.Finished    1 of    1.
--- Stats class ---
nirs.core.ChannelStats
--- Stats.beta ---
  -4.5097e-07




In [23]:
out = eng.evalc(r"""
try
    cfg = jsondecode(fileread(getenv('FNIRS_BUNDLE_JSON')));
    S = load(string(cfg.input_mat_files{1}));

    ch = 1;
    times_s = double(S.times_s(:));
    stim_onsets_s = double(S.stim_onsets_s(:));
    stim_durations_s = double(S.stim_durations_s(:));
    stim_amplitudes = double(S.stim_amplitudes(:));
    yi = double(S.Y(:, ch));
    chrom = char(string(S.chromophores{ch}));

    nuisance_values = [];
    nuisance_n_reg = [];
    nuisance_names = {};
    if isfield(S, 'nuisance_values')
        nuisance_values = double(S.nuisance_values);
    end
    if isfield(S, 'nuisance_n_reg')
        nuisance_n_reg = double(S.nuisance_n_reg(:));
    end
    if isfield(S, 'nuisance_names')
        nuisance_names = S.nuisance_names;
    end

    probe = nirs.core.Probe(zeros(1,3), zeros(1,3), ...
        table(1, 1, {char(chrom)}, 'VariableNames', {'source','detector','type'}));
    data_obj = nirs.core.Data(yi(:), times_s(:), probe);

    task = nirs.design.StimulusEvents();
    task.name = 'task';
    task.onset = stim_onsets_s(:);
    task.dur = stim_durations_s(:);
    task.amp = stim_amplitudes(:);
    data_obj.stimulus('task') = task;

    if ~isempty(nuisance_values) && ~isempty(nuisance_n_reg)
        n_nuis = nuisance_n_reg(ch);
        for k = 1:n_nuis
            reg_vec = double(squeeze(nuisance_values(:, k, ch)));
            if isempty(reg_vec)
                continue;
            end
            reg_name = sprintf('nuis_%02d', k);
            stim_vec = nirs.design.StimulusVector();
            stim_vec.name = reg_name;
            stim_vec.time = times_s(:);
            stim_vec.vector = reg_vec(:);
            try
                stim_vec.regressor_no_interest = true;
            catch
            end
            data_obj.stimulus(reg_name) = stim_vec;
        end
    end

    resample_fs = 4.0;
    if isfield(S, 'matlab_resample_fs')
        resample_fs = double(S.matlab_resample_fs(1));
    end

    resample_job = nirs.modules.Resample();
    resample_job.Fs = resample_fs;
    data_rs = resample_job.run(data_obj);

    glm_job = nirs.modules.GLM();
    glm_job.type = 'AR-IRLS';

    basis_dict = Dictionary();
    hrf_model = lower(char(string(S.hrf_model)));
    switch hrf_model
        case {'glover','canonical'}
            basis_dict('default') = nirs.design.basis.Canonical();
        case 'gamma'
            basis_dict('default') = nirs.design.basis.Gamma();
        otherwise
            error('Unsupported MATLAB HRF model for module workflow: %s', hrf_model);
    end
    glm_job.basis = basis_dict;

    Stats = glm_job.run(data_rs);

    disp('--- Stats.variables ---');
    disp(Stats.variables);
    disp('--- variable names ---');
    disp(Stats.variables.Properties.VariableNames);

    % Inline local_find_task_row
    row_idx = 1;
    vars = Stats.variables;
    var_names = vars.Properties.VariableNames;
    cond_col = '';
    candidates = {'cond','condition','Condition','Cond','variable','Variable'};
    for ii = 1:numel(candidates)
        if any(strcmp(var_names, candidates{ii}))
            cond_col = candidates{ii};
            break;
        end
    end
    disp('--- chosen condition column ---');
    disp(cond_col);

    if ~isempty(cond_col)
        cond_values = string(vars.(cond_col));
        disp('--- cond_values ---');
        disp(cond_values);
        match = find(cond_values == "task", 1, 'first');
        if isempty(match)
            match = find(contains(lower(cond_values), 'task'), 1, 'first');
        end
        if ~isempty(match)
            row_idx = double(match);
        end
    end

    disp('--- row_idx ---');
    disp(row_idx);

    beta_value = double(Stats.beta(row_idx));
    t_value = double(Stats.tstat(row_idx));
    p_value = double(Stats.p(row_idx));

    covb = Stats.covb;
    if ndims(covb) == 2
        se_value = sqrt(double(covb(row_idx, row_idx)));
    elseif ndims(covb) == 3
        se_value = sqrt(double(covb(row_idx, row_idx, 1)));
    else
        se_value = sqrt(double(covb(row_idx, row_idx, 1, 1)));
    end

    dfe = double(Stats.dfe);
    if numel(dfe) > 1
        dfe = dfe(1);
    end

    disp('--- extracted stats ---');
    disp(beta_value);
    disp(se_value);
    disp(t_value);
    disp(p_value);
    disp(dfe);

    T = table( ...
        string("Subj94"), string("hrf_20"), 20, ...
        string("LocalSS_Glover_ARIRLS"), string("matlab_arirls"), ...
        string("glover"), string("AR-IRLS"), ...
        string("test_channel"), string("test_pair"), string(chrom), ...
        string("target"), string("task"), ...
        beta_value, se_value, t_value, p_value, dfe, 0, 0, ...
        'VariableNames', {'subject', 'file_label', 'amplitude_value', 'pipeline_label', 'backend', ...
                          'hrf_model', 'solver', 'channel_name', 'pair_name', 'chromophore', ...
                          'target_status', 'task_regressor', 'beta', 'se', 't_value', 'p_value', ...
                          'dfe', 'shift_index', 'shift_s'});

    out_csv = fullfile(tempdir, 'arirls_single_row_debug.csv');
    writetable(T, out_csv);
    disp('--- wrote csv ---');
    disp(out_csv);

catch ME
    disp(getReport(ME, 'extended'));
end
""", nargout=1)

print(out)

High collinearity: cond(X) = 9291.4322.
.Finished    1 of    1.
--- Stats.variables ---
    <strong>source</strong>    <strong>detector</strong>     <strong>type</strong>        <strong>cond</strong>  
    <strong>______</strong>    <strong>________</strong>    <strong>_______</strong>    <strong>________</strong>

      1          1        {'hbo'}    {'task'}

--- variable names ---
    {'source'}    {'detector'}    {'type'}    {'cond'}

--- chosen condition column ---
cond
--- cond_values ---
task
--- row_idx ---
     1

--- extracted stats ---
  -4.5097e-07

   2.9367e-06

   -0.1536

    0.8780

   2.5696e+03

--- wrote csv ---
/tmp/arirls_single_row_debug.csv



In [24]:
out = eng.evalc(r"""
try
    cfg = jsondecode(fileread(getenv('FNIRS_BUNDLE_JSON')));
    S = load(string(cfg.input_mat_files{1}));

    times_s = double(S.times_s(:));
    stim_onsets_s = double(S.stim_onsets_s(:));
    stim_durations_s = double(S.stim_durations_s(:));
    stim_amplitudes = double(S.stim_amplitudes(:));
    Y = double(S.Y);

    channel_names = S.channel_names;
    chromophores = S.chromophores;

    nuisance_values = [];
    nuisance_n_reg = [];
    nuisance_names = {};
    if isfield(S, 'nuisance_values')
        nuisance_values = double(S.nuisance_values);
    end
    if isfield(S, 'nuisance_n_reg')
        nuisance_n_reg = double(S.nuisance_n_reg(:));
    end
    if isfield(S, 'nuisance_names')
        nuisance_names = S.nuisance_names;
    end

    resample_fs = 4.0;
    if isfield(S, 'matlab_resample_fs')
        resample_fs = double(S.matlab_resample_fs(1));
    end

    hrf_model = lower(char(string(S.hrf_model)));

    n_channels = size(Y, 2);
    disp(['n_channels = ' num2str(n_channels)]);

    for ch = 1:n_channels
        disp(' ');
        disp(['===== CHANNEL ' num2str(ch) ' =====']);
        disp(channel_names{ch});
        try
            yi = double(Y(:, ch));
            chrom = char(string(chromophores{ch}));

            probe = nirs.core.Probe(zeros(1,3), zeros(1,3), ...
                table(1, 1, {char(chrom)}, 'VariableNames', {'source','detector','type'}));
            data_obj = nirs.core.Data(yi(:), times_s(:), probe);

            task = nirs.design.StimulusEvents();
            task.name = 'task';
            task.onset = stim_onsets_s(:);
            task.dur = stim_durations_s(:);
            task.amp = stim_amplitudes(:);
            data_obj.stimulus('task') = task;

            if ~isempty(nuisance_values) && ~isempty(nuisance_n_reg)
                n_nuis = nuisance_n_reg(ch);
                for k = 1:n_nuis
                    reg_vec = double(squeeze(nuisance_values(:, k, ch)));
                    if isempty(reg_vec)
                        continue;
                    end
                    reg_name = sprintf('nuis_%02d', k);
                    stim_vec = nirs.design.StimulusVector();
                    stim_vec.name = reg_name;
                    stim_vec.time = times_s(:);
                    stim_vec.vector = reg_vec(:);
                    try
                        stim_vec.regressor_no_interest = true;
                    catch
                    end
                    data_obj.stimulus(reg_name) = stim_vec;
                end
            end

            resample_job = nirs.modules.Resample();
            resample_job.Fs = resample_fs;
            data_rs = resample_job.run(data_obj);

            glm_job = nirs.modules.GLM();
            glm_job.type = 'AR-IRLS';

            basis_dict = Dictionary();
            switch hrf_model
                case {'glover','canonical'}
                    basis_dict('default') = nirs.design.basis.Canonical();
                case 'gamma'
                    basis_dict('default') = nirs.design.basis.Gamma();
                otherwise
                    error('Unsupported MATLAB HRF model for module workflow: %s', hrf_model);
            end
            glm_job.basis = basis_dict;

            Stats = glm_job.run(data_rs);
            disp('GLM OK');

            vars = Stats.variables;
            disp('variables OK');
            var_names = vars.Properties.VariableNames;
            disp(var_names);

            row_idx = 1;
            cond_col = '';
            candidates = {'cond','condition','Condition','Cond','variable','Variable'};
            for ii = 1:numel(candidates)
                if any(strcmp(var_names, candidates{ii}))
                    cond_col = candidates{ii};
                    break;
                end
            end
            disp(['cond_col = ' cond_col]);

            if ~isempty(cond_col)
                cond_values = string(vars.(cond_col));
                match = find(cond_values == "task", 1, 'first');
                if isempty(match)
                    match = find(contains(lower(cond_values), 'task'), 1, 'first');
                end
                if ~isempty(match)
                    row_idx = double(match);
                end
            end
            disp(['row_idx = ' num2str(row_idx)]);

            beta_value = double(Stats.beta(row_idx));
            disp(['beta = ' num2str(beta_value)]);

        catch ME
            disp('*** FAILURE ***');
            disp(getReport(ME, 'extended'));
            break;
        end
    end

catch ME
    disp(getReport(ME, 'extended'));
end
""", nargout=1)

print(out)

n_channels = 12
 
===== CHANNEL 1 =====
S13_D26 hbo
High collinearity: cond(X) = 9291.4322.
.Finished    1 of    1.
GLM OK
variables OK
    {'source'}    {'detector'}    {'type'}    {'cond'}

cond_col = cond
row_idx = 1
beta = -4.5097e-07
 
===== CHANNEL 2 =====
S13_D26 hbr
High collinearity: cond(X) = 15761.0442.
.Finished    1 of    1.
GLM OK
variables OK
    {'source'}    {'detector'}    {'type'}    {'cond'}

cond_col = cond
row_idx = 1
beta = 1.8531e-06
 
===== CHANNEL 3 =====
S14_D31 hbo
High collinearity: cond(X) = 9291.4322.
.Finished    1 of    1.
GLM OK
variables OK
    {'source'}    {'detector'}    {'type'}    {'cond'}

cond_col = cond
row_idx = 1
beta = 3.3707e-06
 
===== CHANNEL 4 =====
S14_D31 hbr
High collinearity: cond(X) = 15761.0442.
.Finished    1 of    1.
GLM OK
variables OK
    {'source'}    {'detector'}    {'type'}    {'cond'}

cond_col = cond
row_idx = 1
beta = -2.9399e-06
 
===== CHANNEL 5 =====
S5_D4 hbo
High collinearity: cond(X) = 9291.4322.
.Finished    1 of 

In [31]:
from pathlib import Path
import json
import pandas as pd

real_bundle = Path("~/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/matlab_bundle.json").expanduser()
assert real_bundle.exists(), real_bundle

cfg = json.loads(real_bundle.read_text())
print("n input files:", len(cfg["input_mat_files"]))
for i, path in enumerate(cfg["input_mat_files"], start=1):
    print(i, path)

n input files: 6
1 /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0001__NoSS_Glover_ARIRLS__observed__input.mat
2 /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0002__LocalSS_Glover_ARIRLS__observed__input.mat
3 /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0003__LocalSS_Gamma_ARIRLS__observed__input.mat
4 /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0004__NoSS_Glover_ARIRLS__shift001__input.mat
5 /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0005__LocalSS_Glover_ARIRLS__shift001__input.mat
6 /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0006__LocalSS_Gamma_ARIRLS__shift001__input.mat


In [32]:
mini_cfg = {
    "input_mat_files": [cfg["input_mat_files"][0]],
    "output_csv_files": [str(real_bundle.parent / "debug_single_output.csv")],
}

mini_bundle = real_bundle.parent / "debug_single_bundle.json"
mini_bundle.write_text(json.dumps(mini_cfg), encoding="utf-8")

print(mini_bundle)
print(json.dumps(mini_cfg, indent=2))

/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/debug_single_bundle.json
{
  "input_mat_files": [
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/0001__NoSS_Glover_ARIRLS__observed__input.mat"
  ],
  "output_csv_files": [
    "/home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/debug_single_output.csv"
  ]
}


In [27]:
helper_m = Path("~/fnirs-representation-learning/v2/analyzir_arirls_batch.m").expanduser()
assert helper_m.exists(), helper_m

_, matlab_engine = bench.optional_import_matlab_engine()
eng = matlab_engine.start_matlab(config.matlab_startup_options)

helper_dir = helper_m.parent.as_posix()
eng.addpath(helper_dir, nargout=0)
eng.setenv("FNIRS_BUNDLE_JSON", str(mini_bundle), nargout=0)
eng.setenv("FNIRS_ANALYZIR_PATH", str(config.analyzir_path), nargout=0)

print("helper:", eng.eval("which('analyzir_arirls_batch')", nargout=1))
print("GLM:", eng.eval("which('nirs.modules.GLM')", nargout=1))
print("Resample:", eng.eval("which('nirs.modules.Resample')", nargout=1))
print("AR_IRLS:", eng.eval("which('nirs.modules.AR_IRLS')", nargout=1))

helper: /home/asunkari/fnirs-representation-learning/v2/analyzir_arirls_batch.m
GLM: /home/asunkari/nirs-toolbox/+nirs/+modules/GLM.m
Resample: /home/asunkari/nirs-toolbox/+nirs/+modules/Resample.m
AR_IRLS: /home/asunkari/nirs-toolbox/+nirs/+modules/AR_IRLS.m


In [28]:
out = eng.evalc("analyzir_arirls_batch", nargout=1)
print(out)

.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.
.Finished    1 of    1.



In [29]:
debug_csv = real_bundle.parent / "debug_single_output.csv"
print(debug_csv.exists(), debug_csv)

if debug_csv.exists():
    df = pd.read_csv(debug_csv)
    display(df.head())
    print(df[["beta", "se", "t_value", "p_value"]].notna().sum())

True /home/asunkari/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/debug_single_output.csv


,subject,file_label,amplitude_value,pipeline_label,backend,hrf_model,solver,channel_name,pair_name,chromophore,target_status,task_regressor,beta,se,t_value,p_value,dfe,shift_index,shift_s
0,Subj94,no_hrf,0,NoSS_Glover_ARIRLS,matlab_arirls,glover,arirls,S13_D26 hbo,S13_D26,hbo,true_non_target,task,-9.230810e-07,0.000003,-0.305958,0.759661,2576.160846,0,0
1,Subj94,no_hrf,0,NoSS_Glover_ARIRLS,matlab_arirls,glover,arirls,S13_D26 hbr,S13_D26,hbr,true_non_target,task,1.404723e-06,0.000004,0.395281,0.692668,2597.192237,0,0
2,Subj94,no_hrf,0,NoSS_Glover_ARIRLS,matlab_arirls,glover,arirls,S14_D31 hbo,S14_D31,hbo,true_non_target,task,3.322746e-06,0.000002,1.380539,0.167540,2575.845088,0,0
3,Subj94,no_hrf,0,NoSS_Glover_ARIRLS,matlab_arirls,glover,arirls,S14_D31 hbr,S14_D31,hbr,true_non_target,task,-3.433728e-06,0.000002,-1.549886,0.121292,2568.544110,0,0
4,Subj94,no_hrf,0,NoSS_Glover_ARIRLS,matlab_arirls,glover,arirls,S5_D4 hbo,S5_D4,hbo,true_non_target,task,9.920204e-06,0.000005,1.930110,0.053707,2476.412974,0,0


beta       12
se         12
t_value    12
p_value    12
dtype: int64


In [33]:
from pathlib import Path
import json
import pandas as pd

real_bundle = Path("~/fnirs-representation-learning/outputs_benchmark_v7/job_results/Subj94/no_hrf/matlab_inputs/matlab_bundle.json").expanduser()
cfg = json.loads(real_bundle.read_text())

results = []

for i, input_mat in enumerate(cfg["input_mat_files"], start=1):
    mini_output = real_bundle.parent / f"debug_{i:04d}_output.csv"
    mini_bundle = real_bundle.parent / f"debug_{i:04d}_bundle.json"
    mini_cfg = {
        "input_mat_files": [input_mat],
        "output_csv_files": [str(mini_output)],
    }
    mini_bundle.write_text(json.dumps(mini_cfg), encoding="utf-8")

    eng = matlab_engine.start_matlab(config.matlab_startup_options)
    try:
        helper_dir = helper_m.parent.as_posix()
        eng.addpath(helper_dir, nargout=0)
        eng.setenv("FNIRS_BUNDLE_JSON", str(mini_bundle), nargout=0)
        eng.setenv("FNIRS_ANALYZIR_PATH", str(config.analyzir_path), nargout=0)

        out = eng.evalc("analyzir_arirls_batch", nargout=1)

        ok = mini_output.exists()
        n_rows = None
        n_beta = None
        if ok:
            df = pd.read_csv(mini_output)
            n_rows = len(df)
            n_beta = int(df["beta"].notna().sum()) if "beta" in df.columns else None

        results.append({
            "index": i,
            "input_mat": input_mat,
            "ok": ok,
            "n_rows": n_rows,
            "n_nonnull_beta": n_beta,
            "matlab_output_tail": out[-1000:],
        })

    except Exception as exc:
        results.append({
            "index": i,
            "input_mat": input_mat,
            "ok": False,
            "n_rows": None,
            "n_nonnull_beta": None,
            "matlab_output_tail": f"PYTHON/ENGINE EXCEPTION: {type(exc).__name__}: {exc}",
        })
    finally:
        try:
            eng.quit()
        except Exception:
            pass

pd.DataFrame(results)[["index", "input_mat", "ok", "n_rows", "n_nonnull_beta"]]

,index,input_mat,ok,n_rows,n_nonnull_beta
0,1,/home/asunkari/fnirs-representation-learning/o...,True,12,12
1,2,/home/asunkari/fnirs-representation-learning/o...,True,12,12
2,3,/home/asunkari/fnirs-representation-learning/o...,True,12,12
3,4,/home/asunkari/fnirs-representation-learning/o...,True,12,12
4,5,/home/asunkari/fnirs-representation-learning/o...,True,12,12
5,6,/home/asunkari/fnirs-representation-learning/o...,True,12,12


In [34]:
res_df = pd.DataFrame(results)
bad = res_df[(~res_df["ok"]) | (res_df["n_nonnull_beta"].fillna(0) == 0)]
bad[["index", "input_mat", "matlab_output_tail"]]

,index,input_mat,matlab_output_tail


In [35]:
eng.quit()

## Interpretation

- If the bundle builds but the MATLAB subprocess fails, the problem is in the MATLAB helper / AnalyzIR workflow, not in the Python preprocessing.
- If the MATLAB subprocess succeeds and writes rows, then the helper is okay and the next seam to debug is the Python Engine wrapper.
- If you later want to test the Engine wrapper specifically, do it **after** this notebook-level helper test works.
